# Förutsägelse av pallplatser i Formel 1

**Författare:** Felixalfred

I det här projektet undersöker jag om man kan förutsäga vilka förare som tar pallplats (topp 3) i ett Formel 1-lopp, utifrån information som är känd **före** loppet, och vilka faktorer som spelar störst roll.

**Data:** [Formula 1 World Championship (1950-2026)](https://www.kaggle.com/datasets/atharvranjan/formula-1-world-championship-1950-present) från Kaggle.

### Så kör du notebooken
1. Installera paketen: `pip install -r requirements.txt`
2. Öppna notebooken och välj *Run All*. Cellerna är gjorda för att köras uppifrån och ned.

### Innehåll
1. Problem och varför maskininlärning behövs
2. Data och urval
3. Utforskande analys (EDA)
4. Statistiska tester
5. Preprocessing och features
6. Modeller och jämförelse
7. Tolkning av modellen
8. Extra experiment
9. Så fungerar modellen, förklarat utan jargong
10. Slutsats och begränsningar

## 1. Problem och varför maskininlärning behövs

**Frågeställning:** Kan man förutsäga om en förare tar pallplats (plats 1-3) i ett lopp, utifrån sådant som är känt före start, till exempel startposition, kvalresultat samt förarens och stallets form? Vilka faktorer väger tyngst?

**Typ av problem:** Binär klassificering. Målvariabeln är *pallplats* (ja/nej). Klasserna är obalanserade, eftersom bara tre av ungefär tjugo förare hamnar på pallen i varje lopp (ca 15 %).

**Varför räcker det inte med en enkel regel?** En vanlig tumregel är "den som står på pole vinner". Men hur mycket startpositionen betyder varierar mellan banor (det är svårt att köra om på Monaco), mellan stall (en snabb bil kan köra upp från mitten av fältet) och mellan år (regler och bilar förändras). Flera faktorer samverkar dessutom. Maskininlärning kan väga ihop många signaler samtidigt och lära sig dessa samband från historiska lopp. Jag jämför därför modellerna mot just tumregeln, för att visa om maskininlärning faktiskt tillför något.

## 2. Data och urval

Datasetet består av flera tabeller (CSV-filer) som hänger ihop via id-kolumner, till exempel `raceId` och `driverId`. Jag använder dessa:

| Tabell | Innehåll |
|---|---|
| `results` | Resultat per förare och lopp: startposition (`grid`), målposition, poäng, status |
| `races` | Ett lopp per rad: säsong, rundenummer, bana, datum |
| `qualifying` | Kvalresultat per förare och lopp, inklusive tider i Q1, Q2 och Q3 |
| `drivers` | Information om förarna |
| `constructors` | Information om stallen |
| `circuits` | Information om banorna, bland annat koordinater |
| `status` | Förklaring av hur varje förare avslutade loppet (t.ex. i mål, olycka, motorfel) |

Datasetet innehåller också tabellerna `lap_times` och `pit_stops`, men de behövs inte för den här frågeställningen och används därför inte.

### 2.1 Importer och sökväg till datan

Först importerar jag de bibliotek jag behöver:
- `pandas` för att läsa in och hantera tabeller
- `Path` från standardbiblioteket för att hantera filsökvägar på ett sätt som fungerar på både Windows, Mac och Linux

Notebooken ligger i mappen `notebooks/` och datan i `data_raw/`. Därför går sökvägen först upp en nivå (`..`) och sedan in i `data_raw`.

In [1]:
import pandas as pd
from pathlib import Path

# Mappen med rådatan, sett från notebookens plats
DATA = Path("../data_raw")

### 2.2 Funktion för att läsa in en tabell

Jag skriver en liten funktion så att jag slipper upprepa samma rad för varje tabell.

Viktigt här är `na_values="\\N"`. I datasetet står tecknen `\N` där ett värde saknas. Om jag inte berättar det för pandas tolkas `\N` som vanlig text, och då blir hela kolumner (t.ex. `position`) textkolumner i stället för tal. Med inställningen omvandlas `\N` till riktiga saknade värden (`NaN`), som jag kan hantera på ett kontrollerat sätt.

In [2]:
def load(name):
    """Läser in en CSV-fil från data_raw och returnerar den som en pandas-tabell (DataFrame)."""
    return pd.read_csv(DATA / f"{name}.csv", na_values="\\N")

### 2.3 Läs in tabellerna

Varje tabell sparas i en egen variabel med samma namn som filen.

In [ ]:
results       = load("results")
races         = load("races")
qualifying    = load("qualifying")
drivers       = load("drivers")
constructors  = load("constructors")
circuits      = load("circuits")
status        = load("status")

### 2.4 Kontrollera storleken

Det första jag gör efter inläsning är att kontrollera att tabellerna har rimlig storlek. `shape` visar *(antal rader, antal kolumner)*. Det är en enkel kontroll av att inläsningen gått rätt till och att inga filer är tomma eller trasiga.

In [4]:
tabeller = {
    "results": results,
    "races": races,
    "qualifying": qualifying,
    "drivers": drivers,
    "constructors": constructors,
    "circuits": circuits,
    "status": status,
}

for namn, tabell in tabeller.items():
    print(f"{namn:<13} {tabell.shape[0]:>6} rader, {tabell.shape[1]:>2} kolumner")

results        27568 rader, 18 kolumner
races           1164 rader, 18 kolumner
qualifying     11298 rader,  9 kolumner
drivers          879 rader,  9 kolumner
constructors     214 rader,  5 kolumner
circuits          78 rader,  9 kolumner
status           141 rader,  2 kolumner


### 2.5 Titta på de första raderna

`head()` visar de fem första raderna. Det ger en känsla för hur datan ser ut. Här ser jag bland annat att varje rad i `results` är en förare i ett lopp, och att tabellen hänvisar till andra tabeller via id-kolumner (`raceId`, `driverId`, `constructorId`).

In [5]:
results.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22.0,1,1.0,1,1,10.0,58,1:34:50.616,5690616.0,39.0,2.0,1:27.452,218.300,1
1,2,18,2,2,3.0,5,2.0,2,2,8.0,58,+5.478,5696094.0,41.0,3.0,1:27.739,217.586,1
2,3,18,3,3,7.0,7,3.0,3,3,6.0,58,+8.163,5698779.0,41.0,5.0,1:28.090,216.719,1
3,4,18,4,4,5.0,11,4.0,4,4,5.0,58,+17.181,5707797.0,58.0,7.0,1:28.603,215.464,1
4,5,18,5,1,23.0,3,5.0,5,5,4.0,58,+18.014,5708630.0,43.0,1.0,1:27.418,218.385,1


### 2.6 Datatyper och saknade värden

Jag kontrollerar att kolumnerna har rätt datatyp (tal ska vara tal) och hur många värden som saknas. `dtypes` visar datatypen per kolumn och `isna().sum()` räknar saknade värden.

Saknade värden är inte alltid fel. En förare som bröt loppet har till exempel ingen målposition (`position`). Det betyder att tabellen innehåller information i sig, och det tar jag hänsyn till när jag bygger målvariabeln.

In [6]:
results.dtypes

resultId             int64
raceId               int64
driverId             int64
constructorId        int64
number             float64
grid                 int64
position           float64
positionText           str
positionOrder        int64
points             float64
laps                 int64
time                   str
milliseconds       float64
fastestLap         float64
rank               float64
fastestLapTime         str
fastestLapSpeed    float64
statusId             int64
dtype: object

In [7]:
results.isna().sum()

resultId               0
raceId                 0
driverId               0
constructorId          0
number                 6
grid                   0
position           11074
positionText           0
positionOrder          0
points                 0
laps                   0
time               19212
milliseconds       19200
fastestLap         18542
rank               18284
fastestLapTime     18542
fastestLapSpeed    19316
statusId               0
dtype: int64

### 2.7 Vilka säsonger finns i datan, och varför väljer jag 2014-2026?

Datasetet täcker 1950 till 2026, men jag avgränsar analysen till **2014-2026**. Skälen är:

- **Jämförbara regler och bilar.** Säsongen 2014 infördes hybridtekniken med turboladdade V6-motorer, och sedan dess är bilarna och förutsättningarna rimligt jämförbara. Ett lopp från 1970-talet och ett från 2020-talet skiljer sig så mycket i teknik, säkerhet och tillförlitlighet att en modell som lär sig av båda lätt blir svårtolkad. Det är till exempel mycket vanligare att bilar går sönder i äldre lopp.
- **Mer komplett data.** Tabellen `pit_stops` finns bara från 2011, och det gäller flera andra tabeller också. Att börja 2014 ger fullständiga uppgifter för alla de tabeller jag vill kunna använda.
- **Tillräckligt många observationer.** Perioden ger 13 säsonger och drygt 5 400 rader i `results` (en per förare och lopp), vilket räcker för att träna och jämföra flera modeller.
- **Aktuell relevans.** Jag vill kunna testa modellen på de senaste loppen, vilket ligger närmast hur sporten fungerar nu.

**Begränsningar:** Perioden innehåller också regelförändringar, bland annat 2022 (nya aerodynamiska regler) och 2026 (nya tekniska regler, och två nya stall, Audi och Cadillac). Säsongen 2026 är dessutom inte komplett i datan. Jag hanterar det genom att dela upp datan i tid, så att modellen tränas på äldre säsonger och testas på de senaste, och jag tar upp effekten i diskussionen.

Först kollar jag vilka år som finns och hur många lopp varje säsong har.

In [10]:
# Första säsongen som tas med i analysen (hybrid-eran börjar 2014)
FIRST_SEASON = 2014

print("Första säsong i datan:", races["year"].min())
print("Senaste säsong i datan:", races["year"].max())

# Antal lopp per säsong i den period jag analyserar
races[races["year"] >= FIRST_SEASON].groupby("year").size()

Första säsong i datan: 1950
Senaste säsong i datan: 2026


year
2014    19
2015    19
2016    21
2017    20
2018    21
2019    21
2020    17
2021    22
2022    22
2023    22
2024    24
2025    24
2026    15
dtype: int64

### 2.8 Slå ihop tabellerna till en analystabell

Datan ligger i flera tabeller, men för analysen vill jag ha **en tabell med en rad per förare och lopp**. Jag utgår från `results` och lägger till information om loppet (säsong, bana, namn), föraren och stallet. Tabellerna kopplas ihop via id-kolumnerna (`raceId`, `driverId`, `constructorId`) med `merge`.

Två detaljer som gör sammanslagningen säker:
- `how="left"` behåller alla rader från `results`, även om en matchning skulle saknas.
- `validate="many_to_one"` kontrollerar att varje resultatrad matchar **exakt en** rad i den andra tabellen. Om id-kolumnerna skulle innehålla dubbletter hade pandas gett ett felmeddelande i stället för att tyst skapa extra rader.

Jag kontrollerar också att antalet rader är oförändrat efter sammanslagningen. Kolumnerna `points` och `statusId` tar jag med för den utforskande analysen, men de beskriver resultatet av loppet och är därför **inte** tillåtna som features i modellen. Det skulle vara dataläckage.

In [11]:
# Välj ut de kolumner jag behöver och döp om de som har samma namn i olika tabeller
races_small = races[["raceId", "year", "round", "circuitId", "name", "date"]].rename(columns={"name": "race_name"})
drivers_small = drivers[["driverId", "forename", "surname"]]
constructors_small = constructors[["constructorId", "name"]].rename(columns={"name": "constructor_name"})

# Slå ihop: varje resultatrad får info om lopp, förare och stall
df = (
    results
    .merge(races_small, on="raceId", how="left", validate="many_to_one")
    .merge(drivers_small, on="driverId", how="left", validate="many_to_one")
    .merge(constructors_small, on="constructorId", how="left", validate="many_to_one")
)

# Förarens fulla namn i en kolumn
df["driver"] = df["forename"] + " " + df["surname"]

# Behåll bara de kolumner jag behöver
df = df[["raceId", "year", "round", "race_name", "date", "circuitId", "driverId", "driver",
         "constructorId", "constructor_name", "grid", "position", "positionOrder", "points", "statusId"]]

# Kontroll: samma antal rader som i results?
print("Samma antal rader som results:", len(df) == len(results), "-", len(df), "rader")

Samma antal rader som results: True - 27568 rader


### 2.9 Avgränsa till den valda perioden

Nu filtrerar jag bort säsongerna före `FIRST_SEASON` (2014), enligt motiveringen i avsnitt 2.7. `.copy()` gör en egen kopia av den filtrerade tabellen, så att jag senare kan lägga till kolumner utan varningar från pandas.

In [12]:
df = df[df["year"] >= FIRST_SEASON].copy()

print("Rader:", len(df))
print("Säsonger:", df["year"].min(), "-", df["year"].max())
print("Antal lopp:", df["raceId"].nunique())

Rader: 5435
Säsonger: 2014 - 2026
Antal lopp: 267


### 2.10 Skapa målvariabeln: pallplats

Det jag vill förutsäga är om en förare **tar pallplats**, det vill säga slutar på plats 1, 2 eller 3. Jag skapar därför kolumnen `podium` med värdet 1 om `position` är 3 eller lägre, och annars 0.

Förare som inte kom i mål saknar värde i `position` (`NaN`). I en jämförelse som `NaN <= 3` blir resultatet falskt, så de får automatiskt `podium = 0`. Det är rimligt, eftersom en förare som bröt loppet inte kan ha tagit pallplats.

Som kontroll räknar jag antalet pallplatser per lopp. Det ska vara exakt tre i varje lopp. Om något lopp avviker tyder det på ett fel i datan eller i min definition.

In [13]:
# 1 = pallplats (plats 1-3), 0 = ingen pallplats (inklusive de som bröt loppet)
df["podium"] = (df["position"] <= 3).astype(int)

# Kontroll: antal pallplatser per lopp ska vara exakt 3
df.groupby("raceId")["podium"].sum().value_counts()

podium
3    267
Name: count, dtype: int64

### 2.11 Hur ser fördelningen av målvariabeln ut?

Innan jag modellerar vill jag veta hur vanligt det är med pallplats. Det avgör både hur jag ska utvärdera modellerna och vad en enkel gissning skulle ge. Eftersom bara tre av ungefär tjugo förare hamnar på pallen i varje lopp är klasserna **obalanserade**: en modell som alltid gissar "ingen pallplats" skulle ha ungefär 85 % rätt utan att ha lärt sig något. Därför räcker det inte att titta på accuracy (andelen rätt), utan jag använder mått som tar hänsyn till obalansen (t.ex. ROC-AUC, precision och recall).

Jag kollar också hur många förare som startade från grid-position 0. Det betyder oftast start från depån, och de behöver hanteras särskilt i preprocessingen senare.

In [14]:
print("Antal per klass:")
print(df["podium"].value_counts())
print()
print(f"Andel pallplatser: {df['podium'].mean():.1%}")
print()
print("Antal starter med grid = 0:", (df["grid"] == 0).sum())

Antal per klass:
podium
0    4634
1     801
Name: count, dtype: int64

Andel pallplatser: 14.7%

Antal starter med grid = 0: 79
